# 04 向量化与索引构建（抽样验证）

> **目标**：用 `bge-small-en-v1.5` 将阶段 3 的文本块向量化，存入 ChromaDB（余弦相似度），并验证检索与元数据过滤。

> **本 notebook 为抽样验证版**，输入复用 `chunks_sample.jsonl`（1,267 chunks）。**向量库存放在工程目录** `data/chroma_db/`（便于本地验证与 git 备份工程内管理）。全量请用 `vectorize-index-full.ipynb`（外接盘）。

## 存储策略（与第三阶段一致）

| 模式 | Notebook | 输入 | ChromaDB 位置 |
|------|----------|------|---------------|
| **验证** | 本文件 | `data/processed/chunks_sample.jsonl` | **`04 .../data/chroma_db/`**（工程内） |
| **全量** | `vectorize-index-full.ipynb` | `E:\...\oa_comm_chunks.jsonl` | **`E:\...\chroma_db\`**（外接盘） |

## 分割策略 → 向量化衔接

| 项目 | 值 |
|------|-----|
| 嵌入模型 | `BAAI/bge-small-en-v1.5`（384 维） |
| 相似度 | 余弦（`hnsw:space=cosine`） |
| 文档端 | 不加指令前缀 |
| 查询端 | 自动加 BGE 指令前缀 |

## 执行流程

- **C0**: 环境配置 + 设备检测（GPU/CPU）
- **C1**: 加载嵌入模型
- **C2**: 构建 ChromaDB 索引
- **C3**: 保存索引统计
- **C4**: 相似性检索验证
- **C5**: 边界 + 元数据过滤验证

---
## 【C0】环境配置 + 设备检测

检测并记录运行设备（GPU/CPU），作为日后对齐的背景信息。

In [1]:
import sys
import os
import json
from pathlib import Path

# 添加 src 到路径
SRC_DIR = Path("../src").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# === 验证期路径（工程内，不放外接盘）===
INPUT_JSONL = Path("../data/processed/chunks_sample.jsonl")
PERSIST_DIR = Path("../data/chroma_db")          # 向量库（工程目录）
COLLECTION = "pmc_oa_comm_sample"

print(f"输入文件: {INPUT_JSONL.resolve()}")
print(f"向量库目录: {PERSIST_DIR.resolve()} (工程内，验证期)")
print(f"collection: {COLLECTION}")
assert INPUT_JSONL.exists(), f"输入不存在: {INPUT_JSONL}"

输入文件: D:\谷歌\04 向量化与索引构建\data\processed\chunks_sample.jsonl
向量库目录: D:\谷歌\04 向量化与索引构建\data\chroma_db (工程内，验证期)
collection: pmc_oa_comm_sample


In [2]:
# 设备检测（背景信息记录）
import torch

ENV_INFO = {
    "torch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}
if torch.cuda.is_available():
    ENV_INFO["gpu_name"] = torch.cuda.get_device_name(0)
    ENV_INFO["gpu_mem_GB"] = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)

print("=== 运行环境 ===")
for k, v in ENV_INFO.items():
    print(f"  {k}: {v}")

if not ENV_INFO["cuda_available"]:
    print("\n⚠️ 当前为 CPU 版 PyTorch，无法用 GPU。")
    print("   抽样验证（1,267 chunks）用 CPU 即可；全量建议装 CUDA 版 torch。")

=== 运行环境 ===
  torch_version: 2.11.0+cpu
  cuda_available: False
  device: cpu

⚠️ 当前为 CPU 版 PyTorch，无法用 GPU。
   抽样验证（1,267 chunks）用 CPU 即可；全量建议装 CUDA 版 torch。


---
## 【C1】加载嵌入模型

加载 `bge-small-en-v1.5`，确认输出维度 = 384。首次运行会联网下载模型权重（约 130MB）。

In [3]:
from embedder import DocumentEmbedder

embedder = DocumentEmbedder(
    model_name="BAAI/bge-small-en-v1.5",
    batch_size=64,
)

print("模型设备信息:")
for k, v in embedder.device_info().items():
    print(f"  {k}: {v}")
print(f"\n嵌入维度: {embedder.dimension}")
assert embedder.dimension == 384, "bge-small 维度应为 384"

模型设备信息:
  model_name: BAAI/bge-small-en-v1.5
  device: cpu
  torch_version: 2.11.0+cpu
  cuda_available: False


c:\Users\10138\miniconda3\envs\med-rag-verify\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\10138\miniconda3\envs\med-rag-verify\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\10138\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In o


嵌入维度: 384


D:\谷歌\04 向量化与索引构建\src\embedder.py:56: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  return self.model.get_sentence_embedding_dimension()


In [4]:
# 快速验证：文档端 vs 查询端编码
doc_vec = embedder.encode_documents(["Diabetes is a chronic metabolic disease."])
qry_vec = embedder.encode_queries(["What is diabetes?"])
print(f"文档向量维度: {len(doc_vec[0])}")
print(f"查询向量维度: {len(qry_vec[0])}")
print(f"查询端已自动加指令前缀: '{embedder.query_instruction}'")

文档向量维度: 384
查询向量维度: 384
查询端已自动加指令前缀: 'Represent this sentence for searching relevant passages: '


---
## 【C2】构建 ChromaDB 索引

分批读取 chunks，文档端编码后写入 ChromaDB（余弦相似度），支持断点续传。

| 参数 | 说明 |
|------|------|
| `BATCH_SIZE` | 每批编码+入库的 chunk 数 |
| `RESUME` | 断点续传（从 progress.json 续跑） |
| `RESET` | 设 True 会**清空**重建该 collection |

In [5]:
BATCH_SIZE = 256
RESUME = True
RESET = False   # 设 True 删除已有 collection 重建

In [7]:
from index_builder import ChromaIndexBuilder

builder = ChromaIndexBuilder(
    persist_dir=PERSIST_DIR,
    collection_name=COLLECTION,
    embedder=embedder,
)

if RESET:
    builder.client.delete_collection(COLLECTION)
    builder = ChromaIndexBuilder(PERSIST_DIR, COLLECTION, embedder)
    # 同时清除进度文件
    pf = builder._progress_path()
    if pf.exists():
        pf.unlink()
    print("已重置 collection")

print(f"入库前 collection 计数: {builder.collection.count():,}")

入库前 collection 计数: 0


In [8]:
result = builder.build_from_jsonl(
    jsonl_path=INPUT_JSONL,
    batch_size=BATCH_SIZE,
    resume=RESUME,
)
print(f"\n入库完成，collection 共 {result['total_in_collection']:,} 条")

  已入库 1,024 条

入库完成，collection 共 1,267 条


---
## 【C3】保存索引统计

对齐任务书 §2 的 stats 结构，保存到 `outputs/samples/`。

In [9]:
import pandas as pd

# 从输入计算 token 分布（chunk_size_stats）
df = pd.read_json(INPUT_JSONL, lines=True)
token_stats = {
    "mean": round(float(df["token_count"].mean()), 2),
    "max": int(df["token_count"].max()),
    "min": int(df["token_count"].min()),
}

stats = builder.get_stats(chunk_token_stats=token_stats)
stats["data_type"] = "验证样本（1,267 chunks）"
stats["env_info"] = ENV_INFO

out_path = Path("../outputs/samples/index_stats_sample.json")
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(stats, f, indent=2, ensure_ascii=False)

print(f"统计已保存: {out_path.resolve()}")
print(json.dumps(stats, indent=2, ensure_ascii=False))

统计已保存: D:\谷歌\04 向量化与索引构建\outputs\samples\index_stats_sample.json
{
  "collection_name": "pmc_oa_comm_sample",
  "total_chunks": 1267,
  "embedding_model": "BAAI/bge-small-en-v1.5",
  "embedding_dimension": 384,
  "index_built_at": "2026-06-01T15:09:40.056128",
  "distance": "cosine",
  "chunk_size_stats": {
    "mean": 263.22,
    "max": 512,
    "min": 6
  },
  "metadata_fields": [
    "doc_id",
    "chunk_index",
    "total_chunks",
    "source_title",
    "token_count",
    "strategy"
  ],
  "data_type": "验证样本（1,267 chunks）",
  "env_info": {
    "torch_version": "2.11.0+cpu",
    "cuda_available": false,
    "device": "cpu"
  }
}


D:\谷歌\04 向量化与索引构建\src\embedder.py:56: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  return self.model.get_sentence_embedding_dimension()


---
## 【C4】相似性检索验证

1. **自相似性**：从索引取一段文本作查询，应命中自身（距离最小）
2. **语义检索**：用自然语言医学问题检索相关片段

In [10]:
# 自相似性测试：取第一条 chunk 的文本作查询
sample_rec = df.iloc[0]
self_query = sample_rec["text"][:300]

res = builder.query(self_query, n_results=3)
print(f"查询文本（取自 {sample_rec['chunk_id']}）:\n  {self_query[:120]}...\n")
print("=== Top-3 命中 ===")
for i, (cid, dist, doc) in enumerate(zip(
    res["ids"][0], res["distances"][0], res["documents"][0]
)):
    print(f"  [{i+1}] id={cid}  距离={dist:.4f}")
    print(f"      {doc[:100]}...")

hit = res["ids"][0][0] == sample_rec["chunk_id"]
print(f"\n自相似性命中自身: {'✅ 是' if hit else '⚠️ 否'}")

查询文本（取自 PMC176545）:
  The Transcriptome of the Intraerythrocytic Developmental Cycle of Plasmodium falciparum

Plasmodium falciparum is the ca...

=== Top-3 命中 ===
  [1] id=PMC176545  距离=0.1146
      The Transcriptome of the Intraerythrocytic Developmental Cycle of Plasmodium falciparum

Plasmodium ...
  [2] id=PMC514566  距离=0.1705
      In vivo transcriptional profiling of Plasmodium falciparum

Both host and pathogen factors contribut...
  [3] id=PMC514496  距离=0.2421
      Malaria morbidity and immunity among residents of villages with different Plasmodium falciparum tran...

自相似性命中自身: ✅ 是


In [11]:
# 语义检索测试：自然语言医学问题
queries = [
    "What is the role of gene regulation in malaria parasites?",
    "conservation of Asian elephants",
    "circadian rhythm in Drosophila",
]
for q in queries:
    res = builder.query(q, n_results=2)
    print(f"\n查询: {q}")
    for cid, dist, title in zip(
        res["ids"][0], res["distances"][0],
        [m.get("source_title", "") for m in res["metadatas"][0]],
    ):
        print(f"  - [{dist:.3f}] {cid}  {title[:70]}")


查询: What is the role of gene regulation in malaria parasites?
  - [0.205] PMC176545  The Transcriptome of the Intraerythrocytic Developmental Cycle of Plas
  - [0.249] PMC314464  Cell-Passage Activity Is Required for the Malarial Parasite to Cross t

查询: conservation of Asian elephants
  - [0.194] PMC176546  DNA Analysis Indicates That Asian Elephants Are Native to Borneo and A
  - [0.307] PMC449851  Human Population Density and Extinction Risk in the World's Carnivores

查询: circadian rhythm in Drosophila
  - [0.149] PMC193604_chunk1  Drosophila Free-Running Rhythms Require Intercellular Communication
  - [0.158] PMC193604_chunk2  Drosophila Free-Running Rhythms Require Intercellular Communication


---
## 【C5】边界情况 + 元数据过滤验证

In [12]:
# 边界：空查询 / 超长查询
validation = {}

try:
    r_empty = builder.query("", n_results=3)
    validation["空查询"] = f"返回 {len(r_empty['ids'][0])} 条（未报错）"
except Exception as e:
    validation["空查询"] = f"异常: {type(e).__name__}"

try:
    r_long = builder.query("diabetes " * 2000, n_results=3)
    validation["超长查询"] = f"返回 {len(r_long['ids'][0])} 条（截断处理，未报错）"
except Exception as e:
    validation["超长查询"] = f"异常: {type(e).__name__}"

for k, v in validation.items():
    print(f"  {k}: {v}")

  空查询: 返回 3 条（未报错）
  超长查询: 返回 3 条（截断处理，未报错）


In [13]:
# 元数据过滤：只在多块文献（strategy=sliding_window）中检索
res_filter = builder.query(
    "circadian rhythm",
    n_results=3,
    where_filter={"strategy": "sliding_window"},
)
print("过滤条件: strategy = sliding_window")
print(f"返回 {len(res_filter['ids'][0])} 条:")
for cid, meta in zip(res_filter["ids"][0], res_filter["metadatas"][0]):
    print(f"  - {cid}  strategy={meta.get('strategy')}  total_chunks={meta.get('total_chunks')}")

all_sw = all(m.get("strategy") == "sliding_window" for m in res_filter["metadatas"][0])
print(f"\n元数据过滤生效: {'✅ 是' if all_sw else '⚠️ 否'}")

过滤条件: strategy = sliding_window
返回 3 条:
  - PMC193604_chunk3  strategy=sliding_window  total_chunks=4
  - PMC193604_chunk1  strategy=sliding_window  total_chunks=4
  - PMC193604_chunk2  strategy=sliding_window  total_chunks=4

元数据过滤生效: ✅ 是


In [14]:
# 导出验证报告
validation_report = {
    "验证日期": pd.Timestamp.now().isoformat(),
    "数据类型": "验证样本（1,267 chunks）",
    "collection": COLLECTION,
    "向量数量": builder.collection.count(),
    "自相似性命中自身": bool(hit),
    "边界情况": validation,
    "元数据过滤生效": bool(all_sw),
}
rep_path = Path("../outputs/samples/query_validation_sample.json")
with open(rep_path, "w", encoding="utf-8") as f:
    json.dump(validation_report, f, indent=2, ensure_ascii=False)
print(f"验证报告已保存: {rep_path.resolve()}")
print(json.dumps(validation_report, indent=2, ensure_ascii=False))

验证报告已保存: D:\谷歌\04 向量化与索引构建\outputs\samples\query_validation_sample.json
{
  "验证日期": "2026-06-01T15:10:00.961183",
  "数据类型": "验证样本（1,267 chunks）",
  "collection": "pmc_oa_comm_sample",
  "向量数量": 1267,
  "自相似性命中自身": true,
  "边界情况": {
    "空查询": "返回 3 条（未报错）",
    "超长查询": "返回 3 条（截断处理，未报错）"
  },
  "元数据过滤生效": true
}


---
## 完成（抽样验证）

| 产物 | 路径 | Git |
|------|------|-----|
| 向量库（样本 collection） | `04 向量化与索引构建/data/chroma_db/` | ❌ 体积大（`.gitignore` 忽略，可本地重建） |
| 索引统计 | `outputs/samples/index_stats_sample.json` | ✅ |
| 查询验证报告 | `outputs/samples/query_validation_sample.json` | ✅ |

> 抽样流程跑通后，用 **`vectorize-index-full.ipynb`** 处理全量 `oa_comm_chunks.jsonl`，向量库写入外接盘 `E:\med-llm-rag-datasets\chroma_db\`。